# Digit Recognizer (from-scratch NumPy)

Train a small dense network on the Kaggle Digit Recognizer dataset, evaluate on a validation split, and write a submission file.

In [ ]:
import os
from pathlib import Path
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

np.random.seed(42)

## Load Kaggle credentials

Prefers `KAGGLE_API_TOKEN` in the environment; falls back to a local `.config` file (gitignored).

In [ ]:
token = os.environ.get("KAGGLE_API_TOKEN")

if not token:
    config_path = Path(".config")
    if config_path.exists():
        for line in config_path.read_text().splitlines():
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip().strip('"')
            value = value.strip().strip('"')
            if key.upper() == "KAGGLE_API_TOKEN":
                token = value
                break

if not token:
    raise ValueError(
        "No Kaggle token found. Set KAGGLE_API_TOKEN in the environment "
        "or add it to a local .config file."
    )

os.environ["KAGGLE_API_TOKEN"] = token

## Download data (skipped if already present)

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
train_path = data_dir / "train.csv"
test_path = data_dir / "test.csv"

if not train_path.exists() or not test_path.exists():
    import kaggle

    kaggle.api.competition_download_files("digit-recognizer", path=str(data_dir))
    zip_path = data_dir / "digit-recognizer.zip"
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(data_dir)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)
print("train:", train_df.shape, "test:", test_df.shape)
train_df.head(5)

## Train / validation split

Hold out 1,000 labeled examples for validation. Cast labels to `int` and scale pixels to `[0, 1]`.

In [ ]:
data = train_df.to_numpy()
np.random.shuffle(data)

val_size = 1000
val_raw = data[:val_size].T
train_raw = data[val_size:].T

Y_val = val_raw[0].astype(int)
X_val = val_raw[1:] / 255.0

Y_train = train_raw[0].astype(int)
X_train = train_raw[1:] / 255.0

X_test = test_df.to_numpy().T / 255.0

print(f"X_train {X_train.shape}, Y_train {Y_train.shape}")
print(f"X_val   {X_val.shape}, Y_val   {Y_val.shape}")
print(f"X_test  {X_test.shape}")

## Model: 784 → 128 (ReLU) → 10 (softmax)

He initialization, explicit batch size in backprop, mini-batch SGD.

In [ ]:
HIDDEN = 128


def init_params(input_size=784, hidden_size=HIDDEN, output_size=10):
    # He init for ReLU hidden layer; small Gaussian for output layer
    W1 = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
    b1 = np.zeros((hidden_size, 1))
    W2 = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / hidden_size)
    b2 = np.zeros((output_size, 1))
    return W1, b1, W2, b2


def ReLU(Z):
    return np.maximum(0, Z)


def ReLU_deriv(Z):
    return Z > 0


def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)


def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2


def one_hot(Y, num_classes=10):
    Y = Y.astype(int)
    one_hot_Y = np.zeros((num_classes, Y.size))
    one_hot_Y[Y, np.arange(Y.size)] = 1
    return one_hot_Y


def back_prop(Z1, A1, Z2, A2, W2, X, Y):
    m = X.shape[1]
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = (1 / m) * dZ2.dot(A1.T)
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = (1 / m) * dZ1.dot(X.T)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)
    return dW1, db1, dW2, db2


def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1
    W2 = W2 - alpha * dW2
    b2 = b2 - alpha * db2
    return W1, b1, W2, b2


def get_predictions(A2):
    return np.argmax(A2, axis=0)


def get_accuracy(predictions, Y):
    return np.mean(predictions == Y)


def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    return get_predictions(A2)


def gradient_descent(
    X,
    Y,
    X_val,
    Y_val,
    alpha=0.1,
    epochs=50,
    batch_size=64,
):
    W1, b1, W2, b2 = init_params()
    m = X.shape[1]
    history = {"train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        # Shuffle columns (examples) each epoch
        perm = np.random.permutation(m)
        X_shuf = X[:, perm]
        Y_shuf = Y[perm]

        for start in range(0, m, batch_size):
            end = min(start + batch_size, m)
            X_batch = X_shuf[:, start:end]
            Y_batch = Y_shuf[start:end]

            Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X_batch)
            dW1, db1, dW2, db2 = back_prop(Z1, A1, Z2, A2, W2, X_batch, Y_batch)
            W1, b1, W2, b2 = update_params(
                W1, b1, W2, b2, dW1, db1, dW2, db2, alpha
            )

        train_pred = make_predictions(X, W1, b1, W2, b2)
        val_pred = make_predictions(X_val, W1, b1, W2, b2)
        train_acc = get_accuracy(train_pred, Y)
        val_acc = get_accuracy(val_pred, Y_val)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(
                f"Epoch {epoch:3d}  train_acc={train_acc:.4f}  val_acc={val_acc:.4f}"
            )

    return W1, b1, W2, b2, history

## Train

In [ ]:
W1, b1, W2, b2, history = gradient_descent(
    X_train,
    Y_train,
    X_val,
    Y_val,
    alpha=0.1,
    epochs=50,
    batch_size=64,
)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["train_acc"], label="train")
plt.plot(history["val_acc"], label="val")
plt.xlabel("epoch")
plt.ylabel("accuracy")
plt.title("Accuracy over training")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

weights_path = Path("artifacts")
weights_path.mkdir(exist_ok=True)
np.savez(weights_path / "weights.npz", W1=W1, b1=b1, W2=W2, b2=b2)
print("Saved weights to", weights_path / "weights.npz")

## Inspect a validation prediction

Images are reshaped to `(28, 28)` for display.

In [ ]:
def test_prediction(index, W1, b1, W2, b2, X=X_val, Y=Y_val):
    current_image = X[:, index, None]
    prediction = make_predictions(current_image, W1, b1, W2, b2)[0]
    label = Y[index]
    print("Prediction:", prediction)
    print("Label:", label)

    plt.gray()
    plt.imshow(current_image.reshape(28, 28), interpolation="nearest")
    plt.title(f"pred={prediction}, label={label}")
    plt.axis("off")
    plt.show()


test_prediction(10, W1, b1, W2, b2)

## Kaggle submission

In [ ]:
test_predictions = make_predictions(X_test, W1, b1, W2, b2)

submission = pd.DataFrame(
    {
        "ImageId": np.arange(1, len(test_predictions) + 1),
        "Label": test_predictions.astype(int),
    }
)
submission_path = data_dir / "submission.csv"
submission.to_csv(submission_path, index=False)
print("Wrote", submission_path)
submission.head()